# Network-Level Simulation: Community Notes & MMR Misinformation on X

This notebook contains the cleaned network-level simulation code for the project **When Corrections Backfire: The Impact of Community Notes on MMR Misinformation at the User and Network Levels**.

The notebook keeps only the steps reported in the final paper's network-level simulation section:

1. Load a preprocessed user/post-level dataset.
2. Construct user-level misinformation susceptibility and day-0 spike parameters.
3. Simulate misinformation diffusion in a **No-Note world** and a **Note world**.
4. Summarize final prevalence, peak daily spread, total adoption events, and late-period persistence.
5. Run paired statistical tests.
6. Save the final cumulative diffusion figure.

Raw X data, usernames, tweet text, tweet IDs, and API outputs should **not** be shared publicly. This notebook assumes that a private/preprocessed input file is available locally.

## Expected Input File

This notebook expects a preprocessed CSV named:

```text
final_dataset_with_exposure_time.csv
```

The file should contain one row per MMR-related post and include at least these columns:

| Column | Description |
|---|---|
| `author_username` | User identifier. This should be anonymized before any public release. |
| `tweet_id` | Post identifier. Do not share publicly unless fully anonymized or removed. |
| `post_timestamp` | Timestamp of the user's MMR-related post. |
| `first_reply_after_note_visible_created_at` | Timestamp used as the user's inferred exposure time to the Community Note. |
| `final_label` | Final classification label. The notebook treats `MISINFORMATION` as misinformation and all other labels as non-misinformation. |

For GitHub, do **not** upload this CSV if it contains usernames, tweet IDs, post text, or other identifiable information.

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import ttest_rel, wilcoxon

## Configuration

These values match the simulation design reported in the final paper:

- 15,000 simulated nodes
- Watts-Strogatz small-world graph
- Average degree of 10
- Rewiring probability of 0.1
- Event-time window from 62 days before exposure to 62 days after exposure
- Contagion rate of 0.026
- Initial adopters selected from the top 3% of users by baseline misinformation susceptibility

In [ ]:
DATA_PATH = Path("final_dataset_with_exposure_time.csv")
FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(exist_ok=True)

N_NODES = 15_000
AVERAGE_DEGREE = 10
REWIRING_PROBABILITY = 0.1
CONTAGION_RATE = 0.026
SEED_FRACTION = 0.03
EVENT_DAYS = np.arange(-62, 63)

MIN_RUNS = 50
MAX_RUNS = 300
BATCH_SIZE = 10
STABILITY_TOLERANCE = 0.002
STABLE_BATCHES_NEEDED = 2

## Load and Prepare the Preprocessed X Dataset

This section converts timestamps, creates an event-time variable, keeps the 124-day event window, creates a binary misinformation indicator, and restricts the sample to users with at least one MMR-related post before and after inferred Community Note exposure.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. This public notebook expects a private/preprocessed "
        "CSV with user-level exposure timing and misinformation labels. Raw X data is not "
        "included for privacy reasons."
    )

# Load preprocessed dataset.
df = pd.read_csv(DATA_PATH)

# Convert identifiers and timestamps.
df["tweet_id"] = df["tweet_id"].astype(str)
df["post_timestamp"] = pd.to_datetime(df["post_timestamp"], utc=True, errors="coerce")
df["first_reply_after_note_visible_created_at"] = pd.to_datetime(
    df["first_reply_after_note_visible_created_at"], utc=True, errors="coerce"
)

# Drop rows with missing timestamps.
df = df.dropna(
    subset=["post_timestamp", "first_reply_after_note_visible_created_at"]
).copy()

# Event time: days relative to inferred Community Note exposure.
df["days_from_exposure"] = (
    (df["post_timestamp"] - df["first_reply_after_note_visible_created_at"]).dt.total_seconds()
    / 86_400
)
df = df[np.isfinite(df["days_from_exposure"])].copy()

# Keep posts in the 62-day pre/post event window.
# Posts exactly at exposure time are excluded; posts later on day 0 remain in the sample.
df = df[
    ((df["days_from_exposure"] >= -62) & (df["days_from_exposure"] < 0))
    | ((df["days_from_exposure"] > 0) & (df["days_from_exposure"] <= 62))
].copy()

# Before/after period labels.
df["period"] = np.where(df["days_from_exposure"] < 0, "before", "after")

# Integer event day used for event-time summaries.
df["relative_day"] = np.floor(df["days_from_exposure"]).astype(int)

# Binary misinformation indicator.
df["is_misinfo"] = (df["final_label"] == "MISINFORMATION").astype(int)

# Keep users who have at least one post before and after exposure.
user_periods = df.groupby("author_username")["period"].agg(set)
users_with_both_periods = user_periods[
    user_periods.apply(lambda periods: {"before", "after"}.issubset(periods))
].index

df_both = df[df["author_username"].isin(users_with_both_periods)].copy()

print(f"Rows in event window: {len(df):,}")
print(f"Users with both before and after posts: {df_both['author_username'].nunique():,}")
print(f"Rows for users with both periods: {len(df_both):,}")

## Build User-Level Simulation Attributes

The simulation uses two user-level quantities derived from the observed X data:

1. **Baseline susceptibility:** each user's overall misinformation proportion across the full 124-day observation window.
2. **Day-0 spike multiplier:** each user's day-0 misinformation proportion divided by their average pre-exposure daily misinformation proportion.

In the reported simulation, baseline susceptibility is held constant in both worlds. The only difference is that the Note world applies the day-0 spike multiplier on day 0.

In [ ]:
# User-level total and misinformation counts by before/after period.
user_period_summary = (
    df_both.groupby(["author_username", "period"])
    .agg(
        total_posts=("tweet_id", "count"),
        misinfo_posts=("is_misinfo", "sum"),
    )
    .reset_index()
)

user_period_summary["misinfo_prop"] = np.where(
    user_period_summary["total_posts"] > 0,
    user_period_summary["misinfo_posts"] / user_period_summary["total_posts"],
    0,
)

# Convert before/after summaries into one row per user.
user_wide = user_period_summary.pivot(index="author_username", columns="period")
user_wide.columns = [f"{metric}_{period}" for metric, period in user_wide.columns]
user_wide = user_wide.reset_index()

# Overall 124-day misinformation susceptibility.
user_wide["total_posts_124"] = (
    user_wide["total_posts_before"].fillna(0)
    + user_wide["total_posts_after"].fillna(0)
)
user_wide["misinfo_posts_124"] = (
    user_wide["misinfo_posts_before"].fillna(0)
    + user_wide["misinfo_posts_after"].fillna(0)
)
user_wide["baseline_susceptibility"] = np.where(
    user_wide["total_posts_124"] > 0,
    user_wide["misinfo_posts_124"] / user_wide["total_posts_124"],
    0,
)

# Average pre-exposure daily misinformation proportion, coding no-post days as 0.
users = df_both["author_username"].unique()
pre_days = np.arange(-62, 0)
pre_index = pd.MultiIndex.from_product(
    [users, pre_days], names=["author_username", "relative_day"]
).to_frame(index=False)

pre_daily_observed = (
    df_both[df_both["relative_day"] < 0]
    .groupby(["author_username", "relative_day"])
    .agg(total_posts=("tweet_id", "count"), misinfo_posts=("is_misinfo", "sum"))
    .reset_index()
)

pre_daily = pre_index.merge(
    pre_daily_observed, on=["author_username", "relative_day"], how="left"
)
pre_daily[["total_posts", "misinfo_posts"]] = pre_daily[
    ["total_posts", "misinfo_posts"]
].fillna(0)
pre_daily["misinfo_prop_day"] = np.where(
    pre_daily["total_posts"] > 0,
    pre_daily["misinfo_posts"] / pre_daily["total_posts"],
    0,
)

pre_user_baseline = (
    pre_daily.groupby("author_username")["misinfo_prop_day"]
    .mean()
    .reset_index()
    .rename(columns={"misinfo_prop_day": "baseline_daily_misinfo_prop"})
)

# Day-0 misinformation proportion, coding users with no day-0 posts as 0.
day0_index = pd.MultiIndex.from_product(
    [users, [0]], names=["author_username", "relative_day"]
).to_frame(index=False)

day0_observed = (
    df_both[df_both["relative_day"] == 0]
    .groupby(["author_username", "relative_day"])
    .agg(total_posts=("tweet_id", "count"), misinfo_posts=("is_misinfo", "sum"))
    .reset_index()
)

day0_full = day0_index.merge(
    day0_observed, on=["author_username", "relative_day"], how="left"
)
day0_full[["total_posts", "misinfo_posts"]] = day0_full[
    ["total_posts", "misinfo_posts"]
].fillna(0)
day0_full["misinfo_prop_day0"] = np.where(
    day0_full["total_posts"] > 0,
    day0_full["misinfo_posts"] / day0_full["total_posts"],
    0,
)

day0_user = (
    day0_full.groupby("author_username")["misinfo_prop_day0"]
    .mean()
    .reset_index()
)

# Final user attribute table.
user_attrs = (
    user_wide[["author_username", "baseline_susceptibility"]]
    .merge(pre_user_baseline, on="author_username", how="left")
    .merge(day0_user, on="author_username", how="left")
)

user_attrs[["baseline_daily_misinfo_prop", "misinfo_prop_day0"]] = user_attrs[
    ["baseline_daily_misinfo_prop", "misinfo_prop_day0"]
].fillna(0)

# User-specific day-0 spike multiplier.
# If the pre-exposure baseline is zero, use 1.0 to avoid undefined ratios.
user_attrs["spike_multiplier"] = np.where(
    user_attrs["baseline_daily_misinfo_prop"] > 0,
    user_attrs["misinfo_prop_day0"] / user_attrs["baseline_daily_misinfo_prop"],
    1.0,
)
user_attrs["spike_multiplier"] = (
    user_attrs["spike_multiplier"]
    .replace([np.inf, -np.inf], 1.0)
    .fillna(1.0)
)

user_attrs.head()

## Simulation Functions

In [ ]:
def generate_network(n=N_NODES, k=AVERAGE_DEGREE, p_rewire=REWIRING_PROBABILITY, seed=42):
    """Generate a Watts-Strogatz small-world network."""
    np.random.seed(seed)
    return nx.watts_strogatz_graph(n=n, k=k, p=p_rewire, seed=seed)


def assign_empirical_user_attributes(G, user_attrs_df, seed=None):
    """Sample observed user attributes with replacement and assign them to simulated nodes."""
    sampled = user_attrs_df.sample(
        n=G.number_of_nodes(), replace=True, random_state=seed
    ).reset_index(drop=True)

    attrs = {}
    for node, row in zip(G.nodes(), sampled.itertuples(index=False)):
        attrs[node] = {
            "baseline_susceptibility": float(np.clip(row.baseline_susceptibility, 0, 1)),
            "spike_multiplier": float(row.spike_multiplier),
        }

    nx.set_node_attributes(G, attrs)


def initialize_states(G, seed_fraction=SEED_FRACTION):
    """Initialize the top susceptibility nodes as misinformation adopters."""
    states = {node: 0 for node in G.nodes()}
    seed_count = max(1, int(seed_fraction * G.number_of_nodes()))

    seed_nodes = sorted(
        G.nodes(),
        key=lambda node: G.nodes[node]["baseline_susceptibility"],
        reverse=True,
    )[:seed_count]

    for node in seed_nodes:
        states[node] = 1

    return states


def get_daily_params(node_attrs, day, world):
    """Return susceptibility and spike multiplier for a node on a given event day."""
    susceptibility = node_attrs["baseline_susceptibility"]

    if world == "no_note":
        spike_multiplier = 1.0
    elif world == "note":
        spike_multiplier = node_attrs["spike_multiplier"] if day == 0 else 1.0
    else:
        raise ValueError("world must be 'no_note' or 'note'")

    return susceptibility, spike_multiplier


def simulate_event_time_world(G, initial_states, world, eta=CONTAGION_RATE, days=EVENT_DAYS):
    """
    Simulate misinformation diffusion over event time.

    Active nodes remain active after adoption. Each day, active nodes attempt to influence
    inactive neighbors. Adoption probability is eta × susceptibility × spike multiplier.
    """
    states = initial_states.copy()
    daily_total_adopters = []
    daily_new_adopters = []

    for day in days:
        new_states = states.copy()
        new_adopters_today = 0

        for sender in G.nodes():
            if states[sender] != 1:
                continue

            for receiver in G.neighbors(sender):
                if states[receiver] == 1 or new_states[receiver] == 1:
                    continue

                susceptibility, spike_multiplier = get_daily_params(
                    G.nodes[receiver], day, world
                )
                p_adopt = np.clip(eta * susceptibility * spike_multiplier, 0, 1)

                if np.random.rand() < p_adopt:
                    new_states[receiver] = 1
                    new_adopters_today += 1

        states = new_states
        daily_total_adopters.append(sum(states.values()))
        daily_new_adopters.append(new_adopters_today)

    return {
        "days": np.array(days),
        "daily_total_adopters": np.array(daily_total_adopters),
        "daily_new_adopters": np.array(daily_new_adopters),
        "final_prevalence": sum(states.values()) / G.number_of_nodes(),
    }

## Run Paired No-Note vs. Note Simulations

Each run creates a new network, samples user attributes from the observed user-level data, initializes the same starting adopters, then runs both worlds under the same initial conditions.

In [ ]:
def run_event_time_comparison(
    user_attrs_df,
    n_nodes=N_NODES,
    eta=CONTAGION_RATE,
    k=AVERAGE_DEGREE,
    p_rewire=REWIRING_PROBABILITY,
    seed_fraction=SEED_FRACTION,
    min_runs=MIN_RUNS,
    max_runs=MAX_RUNS,
    batch_size=BATCH_SIZE,
    tolerance=STABILITY_TOLERANCE,
    stable_batches_needed=STABLE_BATCHES_NEEDED,
):
    all_total_no_note = []
    all_new_no_note = []
    all_prev_no_note = []
    all_total_note = []
    all_new_note = []
    all_prev_note = []
    all_peak_no_note = []
    all_peak_note = []
    all_volume_no_note = []
    all_volume_note = []
    all_late_gap = []

    prev_mean_note = None
    prev_mean_no_note = None
    stable_batches = 0
    runs_done = 0

    while runs_done < max_runs:
        for i in range(batch_size):
            run_seed = 100 + runs_done + i
            np.random.seed(run_seed)

            G = generate_network(n=n_nodes, k=k, p_rewire=p_rewire, seed=run_seed)
            assign_empirical_user_attributes(G, user_attrs_df, seed=run_seed)
            initial_states = initialize_states(G, seed_fraction=seed_fraction)

            res_no_note = simulate_event_time_world(
                G, initial_states, world="no_note", eta=eta, days=EVENT_DAYS
            )
            res_note = simulate_event_time_world(
                G, initial_states, world="note", eta=eta, days=EVENT_DAYS
            )

            all_total_no_note.append(res_no_note["daily_total_adopters"])
            all_new_no_note.append(res_no_note["daily_new_adopters"])
            all_prev_no_note.append(res_no_note["final_prevalence"])

            all_total_note.append(res_note["daily_total_adopters"])
            all_new_note.append(res_note["daily_new_adopters"])
            all_prev_note.append(res_note["final_prevalence"])

            all_peak_no_note.append(np.max(res_no_note["daily_new_adopters"]))
            all_peak_note.append(np.max(res_note["daily_new_adopters"]))

            all_volume_no_note.append(np.sum(res_no_note["daily_new_adopters"]))
            all_volume_note.append(np.sum(res_note["daily_new_adopters"]))

            late_mask = res_note["days"] >= 30
            late_gap = (
                res_note["daily_total_adopters"][late_mask]
                - res_no_note["daily_total_adopters"][late_mask]
            ).mean()
            all_late_gap.append(late_gap)

        runs_done += batch_size
        current_mean_note = float(np.mean(all_prev_note))
        current_mean_no_note = float(np.mean(all_prev_no_note))

        if runs_done >= min_runs and prev_mean_note is not None and prev_mean_no_note is not None:
            note_change = abs(current_mean_note - prev_mean_note)
            no_note_change = abs(current_mean_no_note - prev_mean_no_note)

            if note_change < tolerance and no_note_change < tolerance:
                stable_batches += 1
            else:
                stable_batches = 0

            if stable_batches >= stable_batches_needed:
                break

        prev_mean_note = current_mean_note
        prev_mean_no_note = current_mean_no_note

    return {
        "days": EVENT_DAYS,
        "runs_done": runs_done,
        "mean_total_no_note": np.mean(all_total_no_note, axis=0),
        "mean_new_no_note": np.mean(all_new_no_note, axis=0),
        "mean_prev_no_note": float(np.mean(all_prev_no_note)),
        "sd_prev_no_note": float(np.std(all_prev_no_note)),
        "mean_total_note": np.mean(all_total_note, axis=0),
        "mean_new_note": np.mean(all_new_note, axis=0),
        "mean_prev_note": float(np.mean(all_prev_note)),
        "sd_prev_note": float(np.std(all_prev_note)),
        "all_prev_no_note": all_prev_no_note,
        "all_prev_note": all_prev_note,
        "all_peak_no_note": all_peak_no_note,
        "all_peak_note": all_peak_note,
        "all_volume_no_note": all_volume_no_note,
        "all_volume_note": all_volume_note,
        "all_late_gap": all_late_gap,
    }

In [ ]:
results = run_event_time_comparison(user_attrs)
print(f"Completed {results['runs_done']} paired simulation runs.")

## Summarize Reported Simulation Metrics

In [ ]:
def summarize_results(results):
    days = results["days"]
    total_no = results["mean_total_no_note"]
    total_note = results["mean_total_note"]
    new_no = results["mean_new_no_note"]
    new_note = results["mean_new_note"]

    final_prev_no = results["mean_prev_no_note"]
    final_prev_note = results["mean_prev_note"]
    final_prev_diff = final_prev_note - final_prev_no

    peak_no = new_no.max()
    peak_note = new_note.max()
    peak_diff = peak_note - peak_no

    volume_no = new_no.sum()
    volume_note = new_note.sum()
    volume_diff = volume_note - volume_no

    diff_over_time = total_note - total_no
    early_diff = diff_over_time[(days >= 0) & (days <= 5)].mean()
    late_diff = diff_over_time[days >= 30].mean()
    final_gap = diff_over_time[-1]

    summary = pd.DataFrame(
        {
            "Metric": [
                "Final misinformation prevalence",
                "Peak daily spread",
                "Total adoption events",
                "Average gap, days 0-5",
                "Average gap, days 30+",
                "Final gap",
            ],
            "No-Note World": [final_prev_no, peak_no, volume_no, np.nan, np.nan, np.nan],
            "Note World": [final_prev_note, peak_note, volume_note, np.nan, np.nan, np.nan],
            "Difference": [
                final_prev_diff,
                peak_diff,
                volume_diff,
                early_diff,
                late_diff,
                final_gap,
            ],
        }
    )
    return summary

summary_table = summarize_results(results)
summary_table

## Statistical Tests

The reported analysis used paired tests because each simulation run produces matched outcomes for the No-Note world and Note world under the same initial conditions.

In [ ]:
def run_stat_tests(name, no_note_values, note_values):
    no_note_values = np.asarray(no_note_values, dtype=float)
    note_values = np.asarray(note_values, dtype=float)
    diff = note_values - no_note_values

    t_stat, t_p = ttest_rel(note_values, no_note_values)

    nonzero_mask = diff != 0
    if nonzero_mask.sum() > 0:
        w_stat, w_p = wilcoxon(note_values[nonzero_mask], no_note_values[nonzero_mask])
    else:
        w_stat, w_p = np.nan, np.nan

    return {
        "Test": name,
        "Mean difference": diff.mean(),
        "t statistic": t_stat,
        "t-test p-value": t_p,
        "Wilcoxon statistic": w_stat,
        "Wilcoxon p-value": w_p,
    }

stat_results = pd.DataFrame(
    [
        run_stat_tests(
            "Final prevalence",
            results["all_prev_no_note"],
            results["all_prev_note"],
        ),
        run_stat_tests(
            "Peak daily spread",
            results["all_peak_no_note"],
            results["all_peak_note"],
        ),
        run_stat_tests(
            "Total diffusion volume",
            results["all_volume_no_note"],
            results["all_volume_note"],
        ),
        run_stat_tests(
            "Late-period gap vs 0",
            np.zeros(len(results["all_late_gap"])),
            results["all_late_gap"],
        ),
    ]
)

stat_results

## Plot and Save Final Figure

In [ ]:
mpl.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
        "font.size": 14,
        "axes.labelsize": 14,
        "axes.titlesize": 14,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 14,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

fig, ax = plt.subplots(figsize=(6.8, 3.6), constrained_layout=True)

ax.plot(
    results["days"],
    results["mean_total_no_note"],
    linewidth=2,
    label="No-Note World",
)
ax.plot(
    results["days"],
    results["mean_total_note"],
    linewidth=2.2,
    label="Note World",
)

ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Days Relative to Community Note Event")
ax.set_ylabel("Mean Cumul. Misinfo Adopters")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", linewidth=0.6, alpha=0.35)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left")

png_path = FIGURE_DIR / "network_diffusion_note_vs_no_note.png"
pdf_path = FIGURE_DIR / "network_diffusion_note_vs_no_note.pdf"
fig.savefig(png_path, dpi=300, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")

plt.show()

print(f"Saved figure to {png_path}")
print(f"Saved figure to {pdf_path}")

## Interpretation

The reported simulation compares a counterfactual **No-Note world**, where susceptibility remains constant and no event-time spike is introduced, against a **Note world**, where the day-0 spike parameter from the user-level analysis is applied. In the final report, the Note world produced higher final misinformation prevalence, higher peak daily spread, and more total adoption events, suggesting that a short-term increase in misinformation sharing can produce a small but persistent increase in network-level diffusion.